# Aggregate daily pairwise statistics

Pool the mergeable daily sufficient statistics for one calendar year. Daily approximate quantiles are intentionally excluded because they cannot be combined into observation-level yearly quantiles.

In [ ]:
from datetime import date
from pathlib import Path

import duckdb
import pandas as pd

repository = Path("..").resolve()

from liljegren_analysis import pooled_difference_statistics

In [ ]:
database_path = repository / "data" / "daily_pairwise.duckdb"
year = 2025
start_date = date(year, 1, 1)
end_date = date(year, 12, 31)

In [ ]:
with duckdb.connect(str(database_path), read_only=True) as connection:
    daily_pairwise = connection.execute(
        """
        SELECT
            result_date,
            implementation_a,
            implementation_b,
            n,
            mean_difference_c,
            variance_difference,
            mean_absolute_difference_c,
            rmse_c,
            min_difference_c,
            max_difference_c,
            max_absolute_difference_c
        FROM daily_pairwise
        WHERE result_date BETWEEN ? AND ?
        ORDER BY result_date, implementation_a, implementation_b
        """,
        [start_date, end_date],
    ).df()

daily_pairwise

## Validate daily coverage

Require one row per implementation pair and date, with complete and consistent coverage across the selected calendar year.

In [ ]:
if daily_pairwise.empty:
    raise ValueError(f"No daily_pairwise rows found for {year}")

pair_columns = ["implementation_a", "implementation_b"]
key_columns = ["result_date", *pair_columns]
if daily_pairwise.duplicated(key_columns).any():
    raise ValueError("daily_pairwise contains duplicate pair/date rows")

expected_dates = pd.date_range(start_date, end_date, freq="D")
actual_dates = pd.DatetimeIndex(daily_pairwise["result_date"].unique()).sort_values()
if not actual_dates.equals(expected_dates):
    missing = expected_dates.difference(actual_dates)
    extra = actual_dates.difference(expected_dates)
    raise ValueError(
        f"Incomplete date coverage: missing={list(missing.date)} "
        f"extra={list(extra.date)}"
    )

coverage = (
    daily_pairwise.groupby(pair_columns, sort=True, as_index=False)
    .agg(date_count=("result_date", "nunique"))
)
if not coverage["date_count"].eq(len(expected_dates)).all():
    raise ValueError("Implementation pairs do not have consistent daily coverage")

coverage

## Pool each implementation pair

Grouping and output construction stay in the notebook. The package helper owns only the statistical formulas and input validation.

In [ ]:
records = []
for (implementation_a, implementation_b), pair in daily_pairwise.groupby(
    pair_columns,
    sort=True,
):
    statistics = pooled_difference_statistics(
        pair["n"],
        mean_differences=pair["mean_difference_c"],
        sample_variances=pair["variance_difference"],
        mean_absolute_differences=pair["mean_absolute_difference_c"],
        root_mean_squared_differences=pair["rmse_c"],
        min_differences=pair["min_difference_c"],
        max_differences=pair["max_difference_c"],
        max_absolute_differences=pair["max_absolute_difference_c"],
    )
    records.append(
        {
            "implementation_a": implementation_a,
            "implementation_b": implementation_b,
            **statistics,
        }
    )

yearly_pairwise = pd.DataFrame.from_records(records)
yearly_pairwise